# Task 2 — Multi-Agent System for Financial Analysis

Build three agents using the "Agents as Tools" pattern:
1. **Budget Agent** — budgeting, spending analysis, financial discipline
2. **Financial Analysis Agent** — stock/portfolio/market analysis via `yfinance`
3. **Orchestrator Agent** — routes queries to the right specialist and synthesizes a single answer

Prereqs: same as Task 1 (`.env` configured, Bedrock model access enabled).

In [1]:
import os

from dotenv import load_dotenv
import yfinance as yf

from strands import Agent, tool
from strands.models import BedrockModel

load_dotenv("../.env")

AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
MODEL_ID = os.getenv("BEDROCK_MODEL_ID", "anthropic.claude-3-5-sonnet-20241022-v2:0")
model = BedrockModel(model_id=MODEL_ID, region_name=AWS_REGION)

## 1. Budget Agent (as a tool)

In [2]:
@tool
def apply_50_30_20_rule(monthly_income: float) -> dict:
    """Split monthly income into needs (50%), wants (30%), and savings (20%)."""
    return {
        "needs": round(monthly_income * 0.5, 2),
        "wants": round(monthly_income * 0.3, 2),
        "savings": round(monthly_income * 0.2, 2),
    }


budget_agent = Agent(
    model=model,
    tools=[apply_50_30_20_rule],
    system_prompt="You help users budget using the 50/30/20 rule and discuss financial discipline.",
)


@tool
def budget_assistant(query: str) -> str:
    """Use this tool for any budgeting, spending, or savings questions."""
    return str(budget_agent(query))

## 2. Financial Analysis Agent (as a tool)

In [3]:
@tool
def get_stock_summary(ticker: str) -> dict:
    """Fetch recent price info for a stock ticker using yfinance."""
    data = yf.Ticker(ticker)
    hist = data.history(period="5d")
    return {
        "ticker": ticker,
        "last_close": float(hist["Close"].iloc[-1]) if not hist.empty else None,
        "five_day_change_pct": (
            round((hist["Close"].iloc[-1] / hist["Close"].iloc[0] - 1) * 100, 2)
            if len(hist) > 1
            else None
        ),
    }


financial_agent = Agent(
    model=model,
    tools=[get_stock_summary],
    system_prompt="You analyze stocks, portfolios, and market trends using live data tools.",
)


@tool
def financial_analysis_assistant(query: str) -> str:
    """Use this tool for stock, portfolio, or market analysis questions."""
    return str(financial_agent(query))

## 3. Orchestrator Agent

In [4]:
orchestrator = Agent(
    model=model,
    tools=[budget_assistant, financial_analysis_assistant],
    system_prompt=(
        "You are a financial orchestrator. Route budgeting questions to budget_assistant "
        "and investing/stock questions to financial_analysis_assistant. Synthesize a single "
        "clear answer for the user, calling both tools if the question spans both topics."
    ),
)

## Test multi-agent routing

In [5]:
response = orchestrator("I earn $7000/month, how should I budget it?")
print(response)


Tool #1: budget_assistant
Sure! Let me apply the 50/30/20 rule to your monthly income right away!
Tool #1: apply_50_30_20_rule
Based on the **50/30/20 rule**, here's how you should budget your **$7,000/month** income:

---

### 📊 Your Monthly Budget Breakdown

| Category | Percentage | Amount |
|---|---|---|
| 🏠 **Needs** | 50% | **$3,500** |
| 🎉 **Wants** | 30% | **$2,100** |
| 💰 **Savings** | 20% | **$1,400** |

---

### 🔍 What Goes Into Each Category?

- 🏠 **Needs ($3,500)** — These are your *essential* expenses:
  - Rent/Mortgage
  - Utilities (electricity, water, internet)
  - Groceries
  - Transportation
  - Health insurance & medical bills
  - Minimum debt payments

- 🎉 **Wants ($2,100)** — These are your *lifestyle* expenses:
  - Dining out & entertainment
  - Subscriptions (Netflix, Spotify, etc.)
  - Shopping & hobbies
  - Vacations & travel

- 💰 **Savings ($1,400)** — This is your *future security*:
  - Emergency fund (aim for 3–6 months of expenses)
  - Retirement contribu

In [6]:
response = orchestrator("How has AAPL performed over the last 5 days?")
print(response)


Tool #2: financial_analysis_assistant

Tool #1: get_stock_summary
Here's a summary of **Apple Inc. (AAPL)** over the last 5 days:

| Metric | Value |
|---|---|
| 📅 Last Close Price | **$333.36** |
| 📉 5-Day Change | **-1.49%** |

### Key Takeaways:
- **AAPL is down approximately 1.49%** over the past 5 trading days, reflecting a modest short-term pullback.
- The stock closed most recently at **$333.36**, suggesting some selling pressure in the near term.
- A ~1.5% dip is relatively minor and could be attributed to broader market fluctuations, profit-taking, or sector rotation rather than any fundamental issue with the company.

> ⚠️ *This is based on recent market data. Always consider broader market context and do your own research before making investment decisions.*

Would you like a deeper analysis, a comparison with other stocks, or insights on a specific aspect of AAPL?Here's a quick snapshot of **Apple Inc. (AAPL)'s** performance over the last 5 trading days:

---

## 🍎 AAPL — 

In [7]:
response = orchestrator(
    "I earn $7000/month, how should I budget it, and how has AAPL performed this week?"
)
print(response)

Sure! Let me fetch both answers for you at the same time! 🚀
Tool #3: budget_assistant

Tool #4: financial_analysis_assistant
Based on the data I already retrieved, here's what we know about **Apple Inc. (AAPL)'s** performance this week:

| Metric | Value |
|---|---|
| 📅 Last Close Price | **$333.36Sure** |
| 📉 ! I already5-Day Change | **-1 have.49%** |

Since your monthly a income from trading your week consists previous message of **.5 trading days**, the Let me fetch the data I already pulled budget directly reflects **this week's performance** — AAPL is down **~1.49%** this week, breakdown closing for you right away!
Tool #2: apply_50_30_20_rule
 at **$333.36**.

> ⚠️ *Note: If the week started more or less than 5 trading days ago (e.g., due to holidays), there may be slight variations. Always verify with a financial platform for precision.*

Would you like to compareIt looks like you AAPL's performance against competitors like've asked ** the same question!MSFT or Here GOOGL**, or

## Next: Task 3

This same orchestrator logic is packaged as a deployable AgentCore entrypoint in
`../agentcore_app/agent.py`. Follow `SETUP_GUIDE.md` section 7 to deploy it with `agentcore`.